<a href="https://colab.research.google.com/github/12thFulminata/llm-foundations/blob/main/01-api-basics/02_gemini_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install -q google-genai

from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

In [11]:
from google.genai import types

SYSTEM = (
    "You are Res Gestae, a Roman history assistant. "
    "Answer in at most two sentences. "
    "End every answer with a citation in the form (Suetonius, <Life> <chapter>). "
    "If you are not sure of the chapter, write (citation unknown)."
)

response = client.models.generate_content(
    model="gemma-4-26b-a4b-it",
    contents="How did Otho die?",
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM,
    ),
)
print(response.text)

ServerError: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}

In [12]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import json

PATH = "/content/drive/MyDrive/Colab Notebooks/res-gestae/suetonius.jsonl"

with open(PATH, encoding="utf-8") as f:
    chapters = [json.loads(line) for line in f]

def get_chapter(life, chapter):
    for c in chapters:
        if c["life"] == life and str(c["chapter"]) == str(chapter):
            return c["text"]
    return None

print(get_chapter("Otho", 11)[:200])

And now being prepared, and just upon the point of dispatching himself, he was induced to suspend the execution of his purpose by a great tumult which had broken out in the camp. Finding that some of 


In [14]:
with_sys = client.models.generate_content(
    model="gemma-4-26b-a4b-it",
    contents="How did Otho die?",
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM,
    ),
)

no_sys = client.models.generate_content(
    model="gemma-4-26b-a4b-it",
    contents="How did Otho die?",
)

print("WITH SYSTEM:\n", with_sys.text, "\n")
print("NO SYSTEM:\n", no_sys.text, "\n")
print("prompt tokens with:", with_sys.usage_metadata.prompt_token_count)
print("prompt tokens without:", no_sys.usage_metadata.prompt_token_count)

ServerError: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}

In [10]:
import time
from google.genai import errors

RETRYABLE = {429, 500, 502, 503, 504}

def ask(prompt, model="gemini-3.5-flash-lite", retries=4, **kwargs):
    for attempt in range(retries):
        try:
            return client.models.generate_content(
                model=model, contents=prompt, **kwargs
            )
        except errors.APIError as e:
            if e.code not in RETRYABLE:
                raise                      # 400, 403, 404...: fail fast
            if attempt == retries - 1:
                raise                      # out of retries: show the real error
            wait = 2 ** attempt
            print(f"{e.code} error, retrying in {wait}s ({attempt + 1}/{retries})")
            time.sleep(wait)

In [15]:
r = ask("How did Otho die?", config=types.GenerateContentConfig(system_instruction=SYSTEM))
print(r.text)

Following his defeat at the Battle of Bedriacum, Otho chose to commit suicide with a dagger rather than prolong the civil war. (Suetonius, Otho 11)


In [ ]:
ask("How did Otho die?", model="gemma-does-not-exist")

In [16]:
for temp in [0.0, 1.5]:
    print(f"=== temperature {temp} ===")
    for i in range(3):
        r = ask(
            "In one sentence, describe Otho.",
            config=types.GenerateContentConfig(temperature=temp),
        )
        print(f"{i + 1}. {r.text.strip()}")

=== temperature 0.0 ===
1. Otho was a Roman emperor who seized power by orchestrating the assassination of Galba, only to commit suicide after a brief three-month reign following his defeat at the Battle of Bedriacum.
503 error, retrying in 1s (1/4)
2. Otho was a Roman emperor who seized power by orchestrating the assassination of Galba, only to commit suicide after a brief three-month reign following his defeat at the Battle of Bedriacum.
3. Otho was the ambitious Roman aristocrat and former friend of Nero who seized power after orchestrating the murder of Galba, only to commit suicide after a brief three-month reign following his defeat by Vitellius in the Year of the Four Emperors.
=== temperature 1.5 ===
1. Otho was the ambitious Roman emperor who ruled for just three months in 69 AD—the infamous "Year of the Four Emperors"—before choosing to commit suicide rather than prolong a bloody civil war.
2. Otho was a notoriously vain and ambitious Roman aristocrat who seized power in the 

In [17]:
r = ask(
    "Describe Otho's life in detail.",
    config=types.GenerateContentConfig(max_output_tokens=4000),
)
print(r.text)
print("finish reason:", r.candidates[0].finish_reason)
print(r.usage_metadata)

**Marcus Salvius Otho** (Ad 32–69) lived one of the most remarkable and transformational lives in Roman history. He transitioned from a hedonistic, debt-ridden playboy and the intimate companion of an emperor to a provincial governor, and finally, to a surprisingly dignified Roman Emperor during the chaotic "Year of the Four Emperors." 

Here is a detailed account of his life, divided into key phases.

---

### 1. Early Life and Rise through Excess (AD 32–58)
Otho was born on April 28, AD 32, into an ancient and noble Etruscan family. His father, Lucius Salvius Otho, had a successful political career and was a favorite of the Emperor Tiberius, largely due to a striking physical resemblance. 

In his youth, Otho was notorious for his extravagance, vanity, and debauchery. He was part of the inner circle of **Nero**, who came to power in AD 54. Otho was unusually obsessed with his appearance; he is said to have plucked his body hair, used bread poultices to prevent facial hair from growin

## Multi-turn chat

In [18]:
r1 = ask("My name is Julius Caesar. Please remember it.")
print(r1.text)

r2 = ask("What is my name?")
print(r2.text)

Salve, Julius Caesar. I have noted your name well. How may I assist you today?
I don't know your name because you haven't told me yet! What would you like me to call you?


In [19]:
history = [
    types.Content(role="user", parts=[types.Part(text="My name is Julius Caesar. Please remember it.")]),
]

r1 = ask(history)
print(r1.text)

history.append(r1.candidates[0].content)
history.append(types.Content(role="user", parts=[types.Part(text="What is my name?")]))

r2 = ask(history)
print(r2.text)
print(len(history), "messages sent")

Hail, Caesar! I shall remember your name. How may I assist you today, dictator of Rome?
Your name is Julius Caesar.
3 messages sent


In [20]:
print("multi-turn prompt tokens:", r2.usage_metadata.prompt_token_count)

single = ask("What is my name?")
print("single prompt tokens:", single.usage_metadata.prompt_token_count)

multi-turn prompt tokens: 39
single prompt tokens: 6


In [21]:
for p in r1.candidates[0].content.parts:
    print("thought:", p.thought, "| has signature:", p.thought_signature is not None)
    print((p.text or "")[:200])
    print("---")

thought: None | has signature: True
Hail, Caesar! I shall remember your name. How may I assist you today, dictator of Rome?
---


In [22]:
answer_parts = [p for p in r1.candidates[0].content.parts if not p.thought]

In [23]:
history = [
    types.Content(role="user", parts=[types.Part(text="My name is Julius Caesar. Please remember it.")]),
]

r1 = ask(history)

answer_parts = [p for p in r1.candidates[0].content.parts if not p.thought]
history.append(types.Content(role="model", parts=answer_parts))
history.append(types.Content(role="user", parts=[types.Part(text="What is my name?")]))

r2 = ask(history)
print(r2.text)
print("multi-turn prompt tokens:", r2.usage_metadata.prompt_token_count)

Your name is Julius Caesar.
multi-turn prompt tokens: 39


In [26]:
import time

start = time.perf_counter()
first = None

stream = client.models.generate_content_stream(
    model="gemma-4-26b-a4b-it",
    contents="In three sentences, describe Otho's coup against Galba.",
)

for chunk in stream:
    if chunk.text:
        if first is None:
            first = time.perf_counter() - start
        print(chunk.text, end="", flush=True)

total = time.perf_counter() - start
print(f"\n\nfirst text: {first:.1f}s | total: {total:.1f}s")

ServerError: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}

## Structured output (Pydantic)

In [25]:
from pydantic import BaseModel

class CitedAnswer(BaseModel):
    answer: str
    life: str
    chapter: int
    quote: str

r = ask(
    "How did Otho die? Cite the Suetonius Life and chapter, and quote the exact supporting sentence.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=CitedAnswer,
        max_output_tokens=3000,
        http_options=types.HttpOptions(timeout=60_000),
    ),
)
print(r.candidates[0].finish_reason)
print(r.text)


a = r.parsed
print(type(a))
print(a.life, a.chapter)


FinishReason.STOP
{
  "answer": "Otho committed suicide by stabbing himself in the chest after deciding to end the civil war and spare his remaining troops.",
  "life": "Otho",
  "chapter": 11,
  "quote": "persecutusque voluntariam mortem demisso in pectus iugulo primam statim lucem hausit."
}
<class '__main__.CitedAnswer'>
Otho 11


In [27]:
try:
    CitedAnswer.model_validate_json(r.text)
except Exception as e:
    print(e)

In [28]:
from typing import Literal

print(sorted({c["life"] for c in chapters}))

['Augustus', 'Caligula', 'Claudius', 'Domitian', 'Galba', 'Julius', 'Nero', 'Otho', 'Tiberius', 'Titus', 'Vespasian', 'Vitellius']


In [29]:
from typing import Literal

Life = Literal[
    "Julius", "Augustus", "Tiberius", "Caligula", "Claudius", "Nero",
    "Galba", "Otho", "Vitellius", "Vespasian", "Titus", "Domitian",
]

class CitedAnswer(BaseModel):
    answer: str
    life: Life
    chapter: int
    quote: str

In [30]:
a = r.parsed
print(a.life, a.chapter)
print(a.quote in get_chapter(a.life, a.chapter))

Otho 11
False


## Baseline

In [38]:
gold = [
    # --- famous (10) ---
    {"id": 1, "type": "famous", "life": "Julius", "chapter": 82,
     "question": "How many stab wounds did Julius Caesar receive when he was assassinated?",
     "answer": "Twenty-three"},
    {"id": 2, "type": "famous", "life": "Julius", "chapter": 32,
     "question": "What did Caesar say as he crossed the Rubicon?",
     "answer": "The die is cast"},
    {"id": 3, "type": "famous", "life": "Augustus", "chapter": 29,
     "question": "What did Augustus boast about the city of Rome's buildings?",
     "answer": "He found it of brick and left it of marble"},
    {"id": 4, "type": "famous", "life": "Augustus", "chapter": 23,
     "question": "What did Augustus cry out after the defeat of Varus?",
     "answer": "Quintilius Varus, give me back my legions!"},
    {"id": 5, "type": "famous", "life": "Augustus", "chapter": 99,
     "question": "What did Augustus ask his friends on his deathbed?",
     "answer": "Whether he had played the comedy of life well, and to applaud"},
    {"id": 6, "type": "famous", "life": "Caligula", "chapter": 55,
     "question": "What honour did Caligula reportedly plan for his horse Incitatus?",
     "answer": "Making him consul"},
    {"id": 7, "type": "famous", "life": "Claudius", "chapter": 10,
     "question": "Where was Claudius found by a soldier after Caligula was murdered?",
     "answer": "Hiding behind a curtain"},
    {"id": 8, "type": "famous", "life": "Nero", "chapter": 49,
     "question": "What did Nero say repeatedly as he prepared to die?",
     "answer": "What an artist dies with me"},
    {"id": 9, "type": "famous", "life": "Vespasian", "chapter": 23,
     "question": "What joke did Vespasian make when he felt death approaching?",
     "answer": "Oh dear, I think I'm becoming a god"},
    {"id": 10, "type": "famous", "life": "Titus", "chapter": 8,
     "question": "What did Titus say on a day when he had done no one a favour?",
     "answer": "Friends, I have lost a day"},

    # --- obscure (15) ---
    {"id": 11, "type": "obscure", "life": "Julius", "chapter": 4,
     "question": "Near which island was the young Caesar captured by pirates?",
     "answer": "Pharmacussa"},
    {"id": 12, "type": "obscure", "life": "Julius", "chapter": 45,
     "question": "How did Caesar try to hide his baldness?",
     "answer": "Combing his hair forward and wearing a laurel wreath"},
    {"id": 13, "type": "obscure", "life": "Augustus", "chapter": 87,
     "question": "What phrase did Augustus use to mean something done very quickly?",
     "answer": "Quicker than asparagus is cooked"},
    {"id": 14, "type": "obscure", "life": "Augustus", "chapter": 101,
     "question": "With whom had Augustus deposited his will?",
     "answer": "The Vestal Virgins"},
    {"id": 15, "type": "obscure", "life": "Tiberius", "chapter": 68,
     "question": "What feat of finger strength is told of Tiberius?",
     "answer": "He could bore through a fresh apple with his finger"},
    {"id": 16, "type": "obscure", "life": "Tiberius", "chapter": 60,
     "question": "How did Tiberius punish the fisherman who surprised him on Capri with a mullet?",
     "answer": "Had his face scrubbed with the fish, then with a crab"},
    {"id": 17, "type": "obscure", "life": "Caligula", "chapter": 50,
     "question": "How many hours a night did Caligula sleep?",
     "answer": "Seldom more than three or four"},
    {"id": 18, "type": "obscure", "life": "Claudius", "chapter": 41,
     "question": "What did Claudius add to the Latin alphabet?",
     "answer": "Three new letters"},
    {"id": 19, "type": "obscure", "life": "Nero", "chapter": 31,
     "question": "What did Nero say when his Golden House was finished?",
     "answer": "That at last he could live like a human being"},
    {"id": 20, "type": "obscure", "life": "Galba", "chapter": 4,
     "question": "What did Augustus say to Galba as a boy while pinching his cheek?",
     "answer": "You too, child, will have a taste of our power"},
    {"id": 21, "type": "obscure", "life": "Otho", "chapter": 12,
     "question": "What did Otho use on his face to keep from growing a beard?",
     "answer": "Moist bread"},
    {"id": 22, "type": "obscure", "life": "Vitellius", "chapter": 13,
     "question": "What did Vitellius call his enormous dish of pike livers, pheasant brains and flamingo tongues?",
     "answer": "The Shield of Minerva"},
    {"id": 23, "type": "obscure", "life": "Domitian", "chapter": 3,
     "question": "How did Domitian spend an hour alone each day early in his reign?",
     "answer": "Catching flies and sticking them with a sharp pin"},
    {"id": 24, "type": "obscure", "life": "Domitian", "chapter": 13,
     "question": "How did Domitian have official letters begin?",
     "answer": "Our Lord and God orders"},
    {"id": 25, "type": "obscure", "life": "Titus", "chapter": 7,
     "question": "What did Titus do with Queen Berenice once he became emperor?",
     "answer": "Sent her away from Rome, against both their wishes"},

    # --- unanswerable (5) ---
    {"id": 26, "type": "unanswerable", "life": None, "chapter": None,
     "question": "What does Suetonius say about Trajan's conquest of Dacia?",
     "answer": None},
    {"id": 27, "type": "unanswerable", "life": None, "chapter": None,
     "question": "According to Suetonius, why did Hadrian build his wall in Britain?",
     "answer": None},
    {"id": 28, "type": "unanswerable", "life": None, "chapter": None,
     "question": "How does Suetonius describe the death of Antinous?",
     "answer": None},
    {"id": 29, "type": "unanswerable", "life": None, "chapter": None,
     "question": "How was the emperor Commodus killed, according to Suetonius?",
     "answer": None},
    {"id": 30, "type": "unanswerable", "life": None, "chapter": None,
     "question": "What does Suetonius say about Septimius Severus' death at York?",
     "answer": None},
]

In [39]:
keywords = {
    1: "twenty", 2: "die", 3: "marble", 4: "give", 5: "applau",
    6: "consul", 7: "behind", 8: "artist", 9: "god", 10: "lost a day",
    11: "pirates", 12: "bald", 13: "asparagus", 14: "vestal", 15: "apple",
    16: "crab", 17: "three hours", 18: "letters", 19: "man", 20: "taste",
    21: "bread", 22: "minerva", 23: "flies", 24: "lord", 25: "berenice",
}

for q in gold:
    if q["life"] is None:
        continue
    text = get_chapter(q["life"], q["chapter"]).lower()
    word = keywords[q["id"]]
    found = word in text
    print(q["id"], "OK " if found else "MISS", word)

1 OK  twenty
2 OK  die
3 OK  marble
4 OK  give
5 OK  applau
6 OK  consul
7 OK  behind
8 OK  artist
9 OK  god
10 OK  lost a day
11 OK  pirates
12 OK  bald
13 OK  asparagus
14 OK  vestal
15 OK  apple
16 OK  crab
17 MISS three hours
18 OK  letters
19 OK  man
20 OK  taste
21 OK  bread
22 OK  minerva
23 OK  flies
24 OK  lord
25 OK  berenice


In [40]:
keywords = {
    1: "twenty", 2: "die", 3: "marble", 4: "give", 5: "applau",
    6: "consul", 7: "behind", 8: "artist", 9: "god", 10: "lost a day",
    11: "pirates", 12: "bald", 13: "asparagus", 14: "vestal", 15: "apple",
    16: "crab", 17: "four hours", 18: "letters", 19: "man", 20: "taste",
    21: "bread", 22: "minerva", 23: "flies", 24: "lord", 25: "berenice",
}

for q in gold:
    if q["life"] is None:
        continue
    text = get_chapter(q["life"], q["chapter"]).lower()
    word = keywords[q["id"]]
    found = word in text
    print(q["id"], "OK " if found else "MISS", word)

1 OK  twenty
2 OK  die
3 OK  marble
4 OK  give
5 OK  applau
6 OK  consul
7 OK  behind
8 OK  artist
9 OK  god
10 OK  lost a day
11 OK  pirates
12 OK  bald
13 OK  asparagus
14 OK  vestal
15 OK  apple
16 OK  crab
17 OK  four hours
18 OK  letters
19 OK  man
20 OK  taste
21 OK  bread
22 OK  minerva
23 OK  flies
24 OK  lord
25 OK  berenice


In [41]:
import json

GOLD_PATH = "/content/drive/MyDrive/Colab Notebooks/res-gestae/gold_v1.jsonl"

with open(GOLD_PATH, "w") as f:
    for q in gold:
        f.write(json.dumps(q) + "\n")

print(sum(1 for _ in open(GOLD_PATH)), "lines saved")

30 lines saved


In [42]:
from typing import Literal, Optional
from pydantic import BaseModel

Life = Literal["Julius", "Augustus", "Tiberius", "Caligula", "Claudius", "Nero",
               "Galba", "Otho", "Vitellius", "Vespasian", "Titus", "Domitian"]

class CitedAnswer(BaseModel):
    answerable: bool
    answer: str
    life: Optional[Life] = None
    chapter: Optional[int] = None
    quote: Optional[str] = None

In [43]:
import time
import httpx
from google.genai import errors

RETRYABLE = {429, 500, 502, 503, 504}

def ask(prompt, model="gemini-3.5-flash-lite", retries=4, **kwargs):
    for attempt in range(retries):
        try:
            return client.models.generate_content(
                model=model, contents=prompt, **kwargs
            )
        except (errors.APIError, httpx.TimeoutException) as e:
            code = getattr(e, "code", "timeout")
            if code != "timeout" and code not in RETRYABLE:
                raise                      # 400, 403, 404...: fail fast
            if attempt == retries - 1:
                raise                      # out of retries: show the real error
            wait = 2 ** attempt
            print(f"{code} error, retrying in {wait}s ({attempt + 1}/{retries})")
            time.sleep(wait)

In [44]:
BASELINE_SYSTEM = (
    "You answer questions using only Suetonius' Lives of the Twelve Caesars. "
    "If the answer is not in that work, set answerable to false and leave "
    "life, chapter and quote empty."
)

for question in ["How did Otho die?",
                 "How was the emperor Commodus killed, according to Suetonius?"]:
    r = ask(
        question,
        config=types.GenerateContentConfig(
            system_instruction=BASELINE_SYSTEM,
            response_mime_type="application/json",
            response_schema=CitedAnswer,
            max_output_tokens=3000,
            http_options=types.HttpOptions(timeout=60_000),
        ),
    )
    print(r.parsed)

answerable=True answer='Otho died by his own hand, stabbing himself in the heart during the civil war after being defeated at Bedriacum.' life='Otho' chapter=11 quote='he died by his own hand, plunging a dagger into his heart under the left breast.'
503 error, retrying in 1s (1/4)
answerable=False answer="The emperor Commodus is not mentioned in Suetonius' Lives of the Twelve Caesars, as the work concludes with Domitian." life=None chapter=None quote=None


In [45]:
BASELINE_CONFIG = types.GenerateContentConfig(
    system_instruction=BASELINE_SYSTEM,
    response_mime_type="application/json",
    response_schema=CitedAnswer,
    max_output_tokens=3000,
    http_options=types.HttpOptions(timeout=60_000),
)

results = []
for q in gold:
    r = ask(q["question"], config=BASELINE_CONFIG)
    a = r.parsed
    if a is None:
        results.append({"id": q["id"], "raw": r.text})
        print(q["id"], "PARSE FAILED")
    else:
        results.append({"id": q["id"], **a.model_dump()})
        print(q["id"], a.answerable, a.life, a.chapter)
    time.sleep(4)

RESULTS_PATH = "/content/drive/MyDrive/Colab Notebooks/res-gestae/baseline_v1_flash-lite.jsonl"
with open(RESULTS_PATH, "w") as f:
    for row in results:
        f.write(json.dumps(row) + "\n")
print(len(results), "results saved")

1 True Julius 82
2 True Julius 32
3 True Augustus 28
4 True Augustus 23
5 True Augustus 99
6 True Caligula 55
7 True Claudius 10
8 True Nero 49
9 True Vespasian 23
503 error, retrying in 1s (1/4)
10 True Titus 8
503 error, retrying in 1s (1/4)
11 True Julius 4
12 True Julius 45
13 True Augustus 87
14 True Augustus 101
15 True Tiberius 68
16 True Tiberius 60
17 True Caligula 50
18 True Claudius 42
19 True Nero 31
20 True Galba 4
21 True Otho 12
503 error, retrying in 1s (1/4)
22 True Vitellius 13
23 True Domitian 3
24 True Domitian 13
25 True Titus 7
26 False None None
503 error, retrying in 1s (1/4)
27 False None None
28 False None None
29 False None None
30 False None None
30 results saved


In [46]:
import pandas as pd

gold_by_id = {q["id"]: q for q in gold}

def score(results):
    rows = []
    for r in results:
        g = gold_by_id[r["id"]]
        row = {"id": r["id"], "type": g["type"]}
        if g["life"] is None:
            row["refused_ok"] = r.get("answerable") is False
        else:
            life_ok = r.get("life") == g["life"]
            ch = r.get("chapter")
            row["life_ok"] = life_ok
            row["chapter_ok"] = life_ok and ch == g["chapter"]
            row["near_miss"] = life_ok and ch is not None and abs(ch - g["chapter"]) == 1
            row["false_refusal"] = r.get("answerable") is False
        rows.append(row)
    df = pd.DataFrame(rows)
    metrics = ["life_ok", "chapter_ok", "near_miss", "false_refusal", "refused_ok"]
    df[metrics] = df[metrics].astype(float)
    return df

df = score(results)
print(df.groupby("type")[["life_ok", "chapter_ok", "near_miss",
                          "false_refusal", "refused_ok"]].mean().round(2))

              life_ok  chapter_ok  near_miss  false_refusal  refused_ok
type                                                                   
famous            1.0        0.90       0.10            0.0         NaN
obscure           1.0        0.93       0.07            0.0         NaN
unanswerable      NaN         NaN        NaN            NaN         1.0
